In [1]:
from ortools.linear_solver import pywraplp

In [ ]:
class Factory:
    def __init__(self, id, capacity, prod_cost):
        self.id = id
        self.capacity = capacity
        self.prod_cost = prod_cost


class Storage:
    def __init__(self, id, capacity, storage_cost=0):
        self.id = id
        self.capacity = capacity
        self.storage_cost = storage_cost

class Orders:
    def __init__(self, id, demand):
        self.id = id
        self.demand = demand

In [ ]:
class Model:
    def __init__(
            self, factories, storages, nodes, orders,
            cost_matrix_f_s, cost_matrix_s_n, cost_matrix_n_o,
            cap_matrix_f_s, cap_matrix_n_o):

        self.factories = factories
        self.storages = storages
        self.nodes = nodes
        self.orders = orders

        self.cost_f_s = cost_matrix_f_s
        self.cost_s_n = cost_matrix_s_n
        self.cost_n_o = cost_matrix_n_o
        self.cap_f_s = cap_matrix_f_s
        self.cap_n_o = cap_matrix_n_o

        self.solver = pywraplp.Solver.CreateSolver("SCIP")

    def build_model(self):

        self.x = {}
        for i, f in enumerate(self.factories):
            for j, s in enumerate(self.storages):
                if self.cost_e_w[i][j] is not None:
                    self.x[(i,j)] = self.solver.NumVar(0, self.solver.infinity(), f'x_{i}_{j}')

        self.y = {}
        for j, s in enumerate(self.storages):
            for k, n in enumerate(self.nodes):
                self.y[(j,k)] = self.solver.NumVar(0, self.solver.infinity(), f'y_{j}_{k}')

        self.z = {}
        for k, n in enumerate(self.nodes):
            for l, o in enumerate(self.orders):
                self.z[(k,l)] = self.solver.NumVar(0, self.solver.infinity(), f'z_{k}_{l}')

        self.use_factories = {}
        for i, f in enumerate(self.factories):
            self.use_factories = self.solver.BoolVar(f'use_factory_{i}')

        self.solver.Add(sum(self.use_factories[i] for i in range(len(self.factories))) <= 2)

        for i, f in enumerate(self.factories):
            flow_from_factory = sum(self.x[(i,j)] for j in range(len(self.storages)) if (i,j) in self.x)
            self.solver.Add(flow_from_factory <= f.capacity)